# 08. 조건별 실행 결과를 비교하고 근거로 설명하기

**목적:** 실행 기록과 영상을 모아 조건 변화의 영향을 비교하고, 완료·실패·중단·미실행을 구분하여 결론의 범위를 정합니다. 결과 읽기 완료와 세 조건 실험 완료를 따로 기록합니다.

예상 시간 60분. [전체 안내](README.md).

![08. 조건별 실행 결과를 비교하고 근거로 설명하기의 전체 과정](../교재그림/vla-evaluation-process.svg)

07에서 로봇을 움직였다면 08은 **“어떤 조건에서 어떤 결과가 나왔으며 무엇까지 말할 수 있는가?”** 를 설명하는 단계입니다. 기본 평가 셀은 기존 파일을 읽고 표를 만들며 정책을 실행하지 않습니다. 추가 실행은 누락 조건을 선택했을 때만 수행합니다.

### 비교할 세 조건과 각각의 질문

| 이름 | 초기 배치 | seed | execute | 비교 질문 |
|---|---:|---:|---:|---|
| A: `seed7-h10` | 0 | 7 | 10 | 기준 실행 |
| B: `seed7-h5` | 0 | 7 | 5 | A와 비교: 실행 길이만 줄이면? |
| C: `init1-h10` | 1 | 7 | 10 | A와 비교: 초기 배치만 바꾸면? |

task·모델·최대 스텝은 동일하게 유지합니다. **B와 C는 실행 길이와 배치가 모두 달라** 차이의 원인을 하나로 말하기 어렵습니다. 같은 조건을 여러 번 반복한 실험도 아니므로 세 조건의 합계 성공 비율을 일반적인 성공률로 확대 해석하지 않습니다.

### 셀을 실행하는 순서

1. 설정 셀에서 SESSION과 `EXPERIMENTS`의 이름·조건을 확인합니다. 07에서 새 이름을 썼다면 실제 파일과 대응시킵니다.
2. **기존 결과 읽기·평가** 셀을 실행합니다. 각 result.json을 읽고 summary.json을 갱신합니다. 모델 추론은 하지 않습니다.
3. 출력의 `missing_names`를 확인합니다. 누락은 아직 결과 파일이 없다는 뜻이며 과제 실패와 다릅니다.
4. 새 실행을 진행할 때만 **누락 실험 실행** 셀의 `RUN_MISSING`을 True로 바꿉니다. 실행은 한 번에 하나씩 이루어집니다. 세 조건이 모두 없으면 세 번의 rollout 시간이 필요합니다.
5. 실행이 끝나면 `RUN_MISSING=False`로 되돌리고 **앞의 읽기·평가 셀을 다시 실행** 합니다. 아래 표 출력만 다시 실행하면 기존 `summary` 변수를 보고 있을 수 있습니다.

기존 미완료 결과도 기록으로 읽습니다. 누락 실행 셀은 result.json이 이미 있는 미완료 실험을 자동 재실행하지 않습니다. 재시도는 기존 자료를 보존하고 새 이름·같은 조건을 EXPERIMENTS에 추가합니다.

### 출력 상태와 분모를 정확하게 읽기

| 구분 | 의미 | 집계 방식 |
|---|---|---|
| 완료·성공 | 정상 종료했고 과제를 달성 | 완료 분모와 성공 분자에 포함 |
| 완료·실패 | 정상 종료했으나 과제 미달성 | 완료 분모에 포함, 성공 분자 제외 |
| 미완료 | 예외·사용자 중단 등 | 별도 incomplete 수로 표시 |
| 결과 누락 | result.json이 없음 | missing_runs로 따로 표시 |

**설명용 예:** 세 조건 중 성공 완료 1개, 실패 완료 1개, 중단 1개라면 완료 기준 비율은 1/2입니다. 반드시 완료 2·중단 1이라는 사실을 함께 제시합니다. 실제로 측정한 숫자가 아닌 집계 예시입니다. 완료가 0이면 비율은 0%가 아니라 `null`/계산 불가입니다.

`summarize(rows)`는 파일이 존재하는 행을 집계합니다. 따라서 `requested_runs`는 현재 코드에서 읽힌 기록 수이며 미실행까지 포함한 계획 조건 수가 아닙니다. 전체 계획은 `len(EXPERIMENTS)`, 누락은 `missing_runs`와 함께 읽습니다. 평가에 inspect/infer를 섞지 않습니다.

### TODO: 숫자 표와 영상 근거를 연결하기

`summary['run_names']`와 `summary['runs']`를 zip으로 대응시켜 각 실행의 task·배치·seed·horizon·status·success·종료 이유·스텝·실행 시간·추론 횟수를 한 행으로 정리합니다. 누락 조건은 별도로 미실행이라고 표시합니다. 값이 없다고 0을 넣으면 실제 측정값처럼 보이므로 미측정으로 남깁니다.

영상에서는 초기 배치와 대상 물체, 실패 또는 차이가 보이는 장면을 기록합니다. JSON의 성공 값만으로 “잘 잡았다”는 과정을 설명하지 말고 영상과 연결합니다. 반대로 영상 인상만으로 환경 판정을 덮어쓰지 않습니다.

**정답 해설:** A↔B는 재관측 전 실행 길이, A↔C는 배치 차이를 탐색합니다. 각 조건 한 번으로 인과관계나 안정적인 성공률을 단정할 수 없습니다. 같은 조건 반복과 추가 배치·seed 실험이 왜 필요한지 `repeat_needed`에 적습니다. 서로 다른 조건의 결과를 한 비율로 합칠 때는 이번 조건 집합의 요약이라는 범위를 명시합니다.

4일차와 비교할 때는 입력, 행동을 정하는 주체, 실패 원인의 세 축으로 설명합니다. 4일차는 웹캠의 손·사람 좌표와 규칙/VLM/좌표 LLM 판단을 사람이 만든 ROS 제어기로 연결했습니다. 오늘은 가상 영상·지시·로봇 상태를 학습된 정책에 넣어 조작 행동을 예측합니다. 과제와 로봇이 달라 성공 숫자만으로 어느 방식이 우월하다고 말할 수 없습니다.


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [5일차 실행 안내](README.md)를 참고합니다.

**5일차 환경:** `nepes-course:physical-ai-vla-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 같은 날에는 준비된 개인 VLA 환경을 유지합니다.

**실행 전 확인 — 컨테이너 내부 Terminal:**

```bash
cd /workspace
/opt/course/bin/python 10.VLA/scripts/preflight.py
```

Python 커널도 `/opt/course/bin/python`을 사용합니다. 강사 배포 `Data/models/smolvla-libero-probe`(LIBERO용 정책 revision `31d453f7edd78c839a8bbc39744a292686daf0de`), `Data/cache/smolvla-probe/huggingface`, `Data/cache/smolvla-probe/libero`와 LIBERO 자산이 필요합니다. 파일 누락은 호스트 개인 Data의 마운트·배포본을 확인합니다. 수강생이 `pip install`, Docker build, 모델 다운로드를 하지 않습니다. CUDA/EGL 문제는 전용 이미지·NVIDIA runtime과 다른 GPU 작업을 확인합니다.

**실행과 저장:** 04부터 08까지의 노트북의 `SESSION = 'session-01'`은 같은 회차이며 결과는 `Data/logs/day5/session-01/<실행이름>/`에 남습니다. 새 비교 전체는 모든 후속 노트북의 SESSION을 함께 `session-02` 등으로 바꿉니다. 04부터 07까지의 노트북의 `run_lesson`은 동일 조건의 `completed` 결과를 읽고 없을 때 실행합니다. 이 평가 노트북은 완료·중단 결과를 먼저 읽고, 누락 실행은 별도 셀에서 선택합니다. 중단 결과는 그대로 집계하며 재시도할 때만 새 실행 이름을 사용합니다. 조건 불일치는 기존 파일을 삭제하거나 성공값으로 바꾸지 말고 원래 조건을 확인합니다. JSON은 있는데 필요한 PNG/NPZ/NPY/영상이 없으면 부분 손상으로 보고 새 회차에서 다시 생성합니다.

**한 번에 한 정책 프로세스만 실행합니다.** 이전 실행이 끝난 뒤 다음 셀을 실행하고 다른 노트북의 infer/rollout을 동시에 시작하지 않습니다. 실행 중단 후에는 해당 Terminal이나 실행 로그에서 자식 프로세스까지 종료됐는지 확인합니다. `status=completed`는 절차 완료, rollout의 `success=true`는 환경 과제 성공이며 inspect/infer의 `success=false`는 아직 작업을 하지 않았다는 뜻입니다.

## MuJoCo 화면 보기: 로컬·원격 공통 noVNC

<!-- procedure:novnc-1744bd:start -->
![noVNC의 정상 대기와 완료](../교재그림/절차/novnc.svg)

03의 준비 검사와 05의 저장 배열 읽기는 새 실험 창을 열지 않습니다. 기존 완료 결과 재사용에도 새 창이 없습니다.
<!-- procedure:novnc-1744bd:end -->


JupyterLab과 **개인 시뮬레이션 화면을 브라우저 탭 두 개로 열어둡니다.** [README의 최초 접속 안내](README.md#novnc-접속)를 따르세요. 기본 배정은 사용자 A: Jupyter 8890 / 화면 8990, 사용자 B: Jupyter 8891 / 화면 8991입니다. 본인 배정 포트를 사용하며 원격은 두 포트의 SSH 터널을 유지합니다. noVNC에서 Connect를 누르고 암호 `nvidia`를 입력하세요. 처음에는 빈 바탕 화면이 정상이며 실험을 실행하면 MuJoCo 창이 나타납니다.

**이 노트북의 기본 평가에는 GUI가 열리지 않습니다.** 기존 결과를 읽고 비교하기 때문입니다. RUN_MISSING=True로 누락 실험을 생성할 때만 실험마다 창이 열립니다. 각 완료 창을 닫아야 다음 조건으로 진행합니다.

설정 셀의 `GUI`는 자동으로 켜집니다. 창 없이 시간 비교를 할 때는 `GUI = False`로 바꾸고 설정 셀을 다시 실행합니다. CLI 대체 실행은 `run_episode.py`에 `--gui`를 붙입니다. 노트북과 CLI를 동시에 실행하지 않습니다.

환경 초기화가 끝난 뒤 실제 초기 장면에서 창이 열립니다. 초기화 중 오류는 노트북 로그에서 확인합니다. 화면에는 실행 이름·모드·현재 상태·스텝·추론 횟수·적용 길이·직전 추론 초가 표시됩니다.

| 화면 상태 | 의미 |
|---|---|
| `INITIAL OBSERVATION` | 초기 관측 준비 완료 |
| `MODEL LOADING` | 모델 로드 중, 환경은 대기 |
| `INFERENCE` | 행동 계산 중, 환경은 대기 |
| `ACTION EXECUTION` | 예측 명령을 환경에 적용 중 |
| `FINISHED` | 종료 이유와 성공 여부 표시. 마지막 장면이며 더 움직이지 않음 |
| `INCOMPLETE / ERROR` | 미완료. 노트북의 원인 로그 확인 |

**정상 종료 화면을 관찰한 뒤 창을 닫으면 셀이 끝납니다.** 결과와 영상은 종료 화면 대기 전에 저장됩니다. 실행 도중 창을 닫으면 다음 안전한 처리 지점에서 사용자 중단으로 기록합니다. GPU 추론·모델 로딩 중에는 그 작업이 반환될 때까지 중단 반영이 늦을 수 있습니다. 창 조작은 표시용 복사본에만 적용되어 정책 카메라와 실제 실험 상태를 바꾸지 않습니다.

**기존 완료 결과를 읽으면 새 GUI는 열리지 않습니다.** 새 시연은 사용하지 않은 실행 이름으로 수행하고 변수·파일 경로도 함께 맞춥니다. GUI 사용 여부는 `result.json`의 `gui_enabled`에 남습니다. 화면 표시 비용이 있으므로 GUI 실행과 비GUI 실행의 시간을 같은 조건처럼 비교하지 않습니다.


**브라우저 탭을 닫아도 실험은 종료되지 않습니다.** 다시 같은 주소로 접속하면 진행 중 화면으로 돌아옵니다. FINISHED 후 화면 안의 MuJoCo 창을 닫아야 셀이 끝납니다. noVNC 연결과 탭은 다음 실험까지 유지하세요. 다른 학생 화면이 보이면 포트를 확인합니다.


## 슬라이드와 함께 진행하기

강사의 목적·흐름 설명을 들은 뒤 해당 셀을 실행하고, 출력과 TODO를 슬라이드의 확인 질문으로 해석합니다. 노트북 사전 정독을 전제하지 않습니다.

쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **조건 비교·평가 요약·TODO·검증** → [5일차 슬라이드 50쪽](https://docs.google.com/presentation/d/1A5eYYMofF0rC7Vmz98XHcc92thRzPhBfkyEKHYWyYYw/edit#slide=id.d5_s046) · [배포 PDF 50쪽](../슬라이드/5일차.pdf#page=50)에서 56쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 이전 자료의 위치·비교 계약·없는 결과 복구

[07_rollout.ipynb의 「예제」와 「TODO」](07_rollout.ipynb)는 `Data/logs/day5/<SESSION>/seed7-h10/result.json`, `seed7-h5/result.json`과 각 rollout.mp4를 만듭니다. 필요한 설정을 다시 쓰면 두 실행은 task0·initial_state0·seed7·최대280스텝이며 execute만 각각10/5입니다. 이 노트북이 추가하는 `init1-h10`은 initial_state=1만 바꾸고 seed7·execute10을 유지합니다.

아래 예제의 **기존 결과 읽기·평가** 셀은 `completed`와 `incomplete`를 그대로 집계하며 정책을 실행하지 않습니다. 누락된 이름은 `missing_names`로 따로 표시합니다. 중단 결과 하나만 있으면 완료0·중단1·완료 기준 성공률 null이며, 결과 폴더가 비어 있으면 완료0·중단0·성공률 null과 누락3개를 표시합니다. 미실행은 중단과 다릅니다.

그다음 **누락 실험 실행(선택)** 셀에서 `RUN_MISSING = True`로 바꾸면 결과 파일이 없는 조건만 한 번에 하나씩 실행합니다. 기존 중단 결과는 덮어쓰거나 재실행하지 않습니다. 실행 후 앞의 읽기·평가 셀을 다시 실행하세요. 중단을 재시도하려면 원래 파일을 보존하고 `EXPERIMENTS`에 `seed7-h10-retry1`처럼 새 이름과 같은 조건을 추가해 원본과 함께 분석합니다. 단순 재시도 때문에 SESSION 전체를 바꾸어 원래 중단을 평가에서 빠뜨리지 마세요.

`result.json`의 status는 절차 완료 여부, success는 환경의 is_success, steps는 실제 적용 행동 수, chunk_seconds는 추론시간 목록, execution_seconds는 준비·로드를 제외한 실행시간입니다. 평가기는 rollout만 받고 `summary.json`을 현재 존재하는 결과와 누락 이름으로 갱신합니다. 영상·초기 PNG가 저장된 조건은 함께 확인하고, 초기 단계 중단으로 없는 자료는 미측정으로 적으세요.

4일차 비교 근거는 [09.Pseudo-VLA/03_ros_vlm_action.ipynb의 「pose 제어와 행동 승인」](../09.Pseudo-VLA/03_ros_vlm_action.ipynb)에 있습니다. 필요한 내용은 USB 손 픽셀→규칙/VLM의 FOLLOW/STOP→사람이 만든 좌표·속도 제어기이며 frame/TTL/인식 오류로 실패한다는 점입니다. 오늘은 가상 영상2·언어·로봇 상태8→학습 정책→행동7의 폐루프이고 입력 분포·전처리·정책·재관측 간격이 실패에 영향을 줍니다. 4일차 실제 로그가 없으면 그날의 정량 비교는 미측정, 위 구조 비교는 개념 답안으로 구분합니다.

## 개념

같은 작업에서 initial_state=0과 1을 비교하고 초기 영상도 확인합니다. horizon 비교는 초기상태·seed를 유지합니다. 비교 조건은 초기0/h10, 초기0/h5, 초기1/h10입니다. 있는 결과와 미실행을 해석하면 결과 읽기 완료이며, 세 조건 모두 completed인 결과·영상을 비교하면 세 조건 실험 완료입니다. 반복 수가 작으므로 일반 성공률로 확대 해석하지 않습니다.

평가표: task, initial_state, seed, horizon, 초기영상 차이, success, 종료이유, 스텝수, 실행시간, 추론횟수. 에러·중단은 완료 실패와 분리합니다. 언어를 임의로 바꿔 새 과제를 풀 것을 요구하지 않습니다.

### 단계별 활동

진행: 비교 조건 정리·추가 실행25분 → 평가표·영상 비교20분 → 짝 설명15분. 초기상태를 바꿀 때 horizon은 유지하고, horizon을 바꿀 때 초기상태는 유지합니다. 완료된 실패와 중단을 한 비율에 섞지 않습니다. 같은 모델이어도 카메라·입력 형식·실행 주기 변화로 실패할 수 있음을 근거와 함께 설명하세요.

In [ ]:
from pathlib import Path
import sys, subprocess, json, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '10.VLA').is_dir())
SCRIPTS = ROOT / '10.VLA/scripts'
sys.path.insert(0, str(SCRIPTS))
from course_runtime import DATA, lesson_runs, run_or_read_lesson, summarize
# 04~08 모두 같은 회차를 사용합니다. 전체 비교를 새로 할 때 session-02 등으로 함께 변경하세요.
GUI = os.environ.get('COURSE_VLA_GUI', '0') == '1'  # 개인 noVNC 화면 사용
SESSION = 'session-01'
RUNS = lesson_runs(DATA, SESSION)
RUNS.mkdir(parents=True, exist_ok=True)
EXPERIMENTS = [
    ("seed7-h10", dict(execute=10)),
    ("seed7-h5", dict(execute=5)),
    ("init1-h10", dict(initial_state=1)),
]


## 예제 — 기존 결과 읽기·평가

<!-- procedure:lesson-44-b129cb:start -->
![있는 결과부터 읽고 누락만 실행](../교재그림/절차/lesson-44.svg)

기본 평가는 파일 읽기입니다. 누락 실행만 새 창이 열리며 FINISHED 후 MuJoCo 창을 닫습니다.
<!-- procedure:lesson-44-b129cb:end -->


이 셀은 정책을 실행하지 않습니다. 누락 실험 실행 후 다시 실행해 요약을 갱신하세요.


In [ ]:
# 기존 완료·중단 기록만 읽습니다. 없는 실험은 실행하지 않고 별도로 표시합니다.
rows, result_names, missing_names = [], [], []
for name, conditions in EXPERIMENTS:
    row = run_or_read_lesson(SCRIPTS, RUNS, 'rollout', name,
                            allow_incomplete=True, run_missing=False, **conditions)
    if row is None:
        missing_names.append(name)
    else:
        rows.append(row)
        result_names.append(name)
summary = {'summary': summarize(rows), 'runs': rows,
           'run_names': result_names, 'missing_runs': missing_names}
(RUNS/'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n')
print('아직 result.json이 없는 실험:', missing_names)
summary['summary']


### 누락 실험 실행 — 세 조건 실험 완료에 필요한 경우

결과 읽기만 진행할 때는 아래 값을 False로 둡니다. 세 조건 실험 완료를 위해 누락된 조건을 실행할 준비가 되면 True로 바꿉니다. 기존 완료·중단 파일은 유지합니다. 실행이 중단되어도 앞의 읽기·평가 셀로 돌아가 새로 남은 result.json을 집계하세요. **실행 후 아래 값을 다시 False로 두고 읽기·평가 셀을 재실행** 합니다. 세 결과가 모두 없으면 정책 rollout 세 번의 시간이 필요합니다.


### 실행 전 확인 — FINISHED 후 MuJoCo 창 닫기

**noVNC에서 `FINISHED`를 확인하면 화면 안의 MuJoCo 창 오른쪽 위 ×를 눌러 닫아주세요.** 창을 닫아야 아래 셀이 완료되고 다음 단계로 진행됩니다. 결과는 이미 저장되었지만, 마지막 장면을 관찰하도록 창을 유지하므로 노트북의 `[*]`와 경과 시간은 계속 표시됩니다. **브라우저 탭만 닫으면 셀은 끝나지 않습니다.** 실행 중 창을 닫으면 중단으로 처리되므로 `FINISHED`를 확인한 뒤 닫습니다.

이 안내는 `RUN_MISSING=True`로 누락 실험을 새로 실행하며 GUI를 사용할 때만 적용됩니다. **각 실험의 완료 창을 닫아야 다음 실험이 시작됩니다.** 마지막 실험의 창까지 닫은 뒤 기존 결과 읽기·평가 셀을 다시 실행합니다. `RUN_MISSING=False`이거나 기존 기록만 평가할 때는 창을 닫으며 기다릴 필요가 없습니다.

기존 완료 결과를 재사용하거나 `GUI=False`로 실행하면 새 GUI 창이 열리지 않으며, 창 닫기를 기다리지 않습니다.


In [ ]:
RUN_MISSING = False
if RUN_MISSING:
    for name, conditions in EXPERIMENTS:
        # 읽기·평가 후 다른 셀에서 생긴 결과도 덮어쓰지 않습니다.
        if not (RUNS/name/'result.json').exists():
            run_or_read_lesson(SCRIPTS, RUNS, 'rollout', name, gui=GUI, **conditions)
    print('실행 완료. 위의 기존 결과 읽기·평가 셀을 다시 실행하세요.')
else:
    print('기존 기록만 평가했습니다. 누락 조건 실행은 RUN_MISSING=True로 선택하세요.')


### 입력과 실행 흐름 확인

In [ ]:
for name, row in zip(summary['run_names'], summary['runs']):
    print('실행 이름:', name)
    print({key: row.get(key) for key in ['initial_state_index','seed','execution_horizon','status','success','steps','execution_seconds']})

## TODO

### 두 완료 기준을 따로 기록하기

| 완료 기준 | 확인할 내용 |
|---|---|
| **결과 읽기 완료** | 계획한 세 조건의 완료·중단·미실행을 구분하고, 있는 JSON·영상의 근거와 미측정 항목·한계를 표와 답안에 기록합니다. 미실행이 있어도 이 기준은 완료할 수 있습니다. |
| **세 조건 실험 완료** | 같은 task·모델·seed7·최대280스텝에서 초기0/h10, 초기0/h5, 초기1/h10의 세 조건이 모두 `status=completed`이고, 각 결과·영상과 조건 비교를 확인합니다. |

`completed`인 과제 실패(`success=false`)도 실험 완료에 포함합니다. `incomplete`인 오류·중단과 결과 파일이 없는 미실행은 세 조건 실험 완료에 포함하지 않습니다. 중단 후 재시도는 새 실행 이름으로 기록하고 원래 중단도 보존합니다. 같은 조건의 완료 결과는 재사용하며 성공할 때까지 반복할 필요는 없습니다.

답안에는 **결과 읽기 완료: 예/아니오**, **세 조건 실험 완료: 예/아니오**와 근거 실행 이름을 각각 적습니다. 결과 읽기만 마친 상태를 세 조건 실험 완료로 표시하지 않습니다.

계획한 세 실험의 현재 결과·중단·미실행 상태를 표로 정리하고 더 많은 반복이 필요한 이유를 쓰세요. Pseudo-VLA와 VLA의 입력·행동 결정·실패 원인을 비교하세요.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**작성 위치와 출력:** 바로 아래 코드 셀의 `notes = {}`에 관찰 답안을 dict로 작성하고 노트북을 저장합니다. 새 정책 함수를 구현하는 TODO가 아닙니다. 코드 검토만 할 때는 아래 형식의 의미만 읽고, 실제로 관찰하지 않은 값은 `None`/`미측정`으로 남깁니다. notes 출력은 파일 저장이 아니므로 답안은 셀에 직접 적고 노트북 자체를 저장하세요.
**입력 자료:** seed7-h10,seed7-h5,init1-h10의 result.json을 모은 summary.json.
**권장 답안 형식** (key 이름은 기록용 권장안이며 자동 채점 schema가 아닙니다):
```python
notes = {'runs': [],
 'repeat_needed': '반복·조건 통제·표본 한계를 설명',
 'pseudo_vla_vs_vla': '입력/행동 결정/실패 원인을 비교',
 'evidence_status': '미측정'}
```
**진행 힌트:** summary["runs"]를 한 행씩 읽어 run 이름, initial_state_index, seed, execution_horizon, status, success, steps, execution_seconds를 표로 만듭니다. 세 실험은 동일 조건 반복3회가 아니라 조건 비교이므로 비율 하나로 일반 성공률을 말하지 않습니다. 선행 두 run의 원래 위치는 07_rollout.ipynb의 예제·TODO입니다. 파일이 없으면 missing_runs에 표시하며 별도 누락 실험 실행 셀에서 생성할 수 있습니다. 기존 incomplete는 보존해 집계하고, 미실행이나 미측정 값을 성공으로 채우지 않습니다.
**강의 멘트:** “명령 실행이 끝난 completed와 물체 과제를 수행한 success가 다릅니다. 실패 이유는 영상과 로그를 같이 봅니다.”
**완료 확인:** 각 항목에 근거와 이유가 있고 실제 관찰/실측 값과 개념 설명을 구분했는지 확인합니다. 정답 문단의 수치는 관찰하지 않은 자기 실행 결과를 대신하지 않습니다.
<!-- todo-review:requirements:end -->

**표에 넣을 열과 출처:** `zip(summary['run_names'], summary['runs'])`로 실제 실행 이름과 결과를 대응시키고 task, initial_state_index, seed, execution_horizon, status, success, termination_reason, steps, execution_seconds, 추론횟수 `len(chunk_seconds)`를 한 행에 기록합니다. 초기영상 차이·실패 발생 시점은 각 run의 image.png/image2.png·rollout.mp4를 읽어 직접 추가합니다. `summary['missing_runs']`의 이름은 미실행 행으로 별도 표시합니다. notes['runs']에는 이 행 dict 목록을 넣거나 노트북 Markdown 표를 작성하고 notes에 표 위치를 적습니다.

A↔B는 horizon만, A↔C는 initial_state만 달라야 합니다. 세 결과의 합계 비율은 이번 조건 비교 요약이며 동일 조건 반복 성공률로 표현하지 않습니다. `repeat_needed`에는 추가할 초기 배치/seed 반복과 그 이유, `pseudo_vla_vs_vla`에는 입력·행동을 정하는 주체·실패 원인 세 축을 모두 씁니다. 실제 오류 결과는 status/error에 따라 미완료로 구분하고 미측정 값을 0 또는 성공으로 대신하지 않습니다.


In [ ]:
# 관찰 결과/실험 설정을 여기에 기록하세요.
notes = {}
notes

## 검증과 정답·복구

4일차 입력은 웹캠 손 영상이며 판단 결과를 규칙 제어기로 연결합니다. 오늘은 가상 카메라·언어·로봇 상태를 작업용 정책에 넣어 행동을 예측합니다. 입력 분포, 전처리, 제어계약, 재관측 빈도 모두 실패에 영향을 줍니다.

TODO 완료 예시(완료 결과가 있으면 같은 조건인지 확인하고 읽습니다):
```python
[(name, r.get('status'), r.get('initial_state_index'), r.get('execution_horizon'), r.get('success'), r.get('steps'))
 for name, r in zip(summary['run_names'], summary['runs'])]
```

파일·캐시·CUDA 오류는 [복구 안내](README.md#복구)와 이 노트북 시작 절에 있습니다. 필요한 복구는 전용 이미지·/opt/course 커널 확인, Data 마운트·배포 모델/캐시 존재 확인, 다른 GPU 작업 종료 후 한 프로세스 실행입니다. 누락 파일을 다운로드하거나 상태8을 잘라 우회하지 않습니다. CLI는 노트북과 같은 실험의 대체 실행입니다. 같은 조건의 완료·중단 결과는 다시 읽습니다. 중단 재시도는 EXPERIMENTS에 새 이름을 추가하여 원본과 함께 평가합니다. 새로운 비교 회차를 시작할 때만 SESSION과 후속 노트북의 회차를 함께 바꾸세요. 실험 출력은 삭제하지 않습니다.

### 정답 해설 — 입력부터 결과까지

**실험표 읽기:** A=초기0/h10, B=초기0/h5, C=초기1/h10을 비교합니다. A↔B는 실행 묶음 길이, A↔C는 초기 배치 차이입니다. B↔C는 둘 다 달라 원인을 하나로 말하기 어렵습니다.

rollout만 작업 성공률 분모에 넣고 inspect/infer를 섞지 않습니다. 정상 완료 실패와 오류·사용자 중단도 구분합니다. 각 조건 한 번의 결과는 탐색 관측이므로 전체 성공률·실물 로봇 성능으로 일반화하지 않습니다.

**확인 질문:** 세 번 모두 성공했을 때에도 추가할 초기 조건·반복 실험이 필요한 이유는 무엇인가요? 각 주장 옆에 자신의 result.json과 영상 근거를 적으세요.

**적용·재확인:** 관찰 답안은 자신의 실제 파일·조건으로 작성합니다. 완료된 동일 조건 결과는 다시 읽고, 새 실험은 새 실행 이름을 사용합니다. `SESSION`을 바꾸면 후속 노트북도 맞춥니다. 형상 검사·실행 완료·작업 성공을 따로 확인하며, 예시 수치를 자기 측정값으로 복사하지 않습니다.

<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 초기배치와 horizon 비교, completed/success/incomplete 분리

[5일차 슬라이드 50쪽](https://docs.google.com/presentation/d/1A5eYYMofF0rC7Vmz98XHcc92thRzPhBfkyEKHYWyYYw/edit#slide=id.d5_s046) · [배포 PDF 50쪽](../슬라이드/5일차.pdf#page=50)로 돌아가 **50–56쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

